# Phase 6 — Evaluation & Comparison

This notebook evaluates and compares all three iris recognition systems on the held-out test set:

| System | Type | Representation |
|---|---|---|
| **ArcFace** | Deep learning (IrisNet + ArcFace loss) | 512-D L2-normalised embeddings |
| **Softmax** | Deep learning (IrisNet + softmax head) | 512-D L2-normalised embeddings |
| **Gabor** | Classical (Gabor filter bank) | 262,144-bit IrisCodes |

### Evaluation protocol
1. Load all test images (4,364 samples across 3,960 identities)
2. Generate genuine pairs (same identity) and impostor pairs (different identity)
3. Extract embeddings/codes from each system
4. Compute similarity scores for all pairs
5. Report EER, TAR@FAR=1%, TAR@FAR=0.1%, ROC curves, DET curves, score distributions

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Disable XLA autotuner (crashes on RTX 5090 Blackwell)
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'

# Set LD_LIBRARY_PATH for pip-installed NVIDIA libraries
NVIDIA_LIB = os.path.join('..', '.venv', 'lib', 'python3.11', 'site-packages', 'nvidia')
if os.path.isdir(NVIDIA_LIB):
    lib_dirs = [
        f'{NVIDIA_LIB}/cudnn/lib', f'{NVIDIA_LIB}/cublas/lib',
        f'{NVIDIA_LIB}/cuda_runtime/lib', f'{NVIDIA_LIB}/cufft/lib',
        f'{NVIDIA_LIB}/cusparse/lib', f'{NVIDIA_LIB}/cusolver/lib',
        f'{NVIDIA_LIB}/nvjitlink/lib', f'{NVIDIA_LIB}/cuda_cupti/lib',
        f'{NVIDIA_LIB}/cuda_nvrtc/lib', f'{NVIDIA_LIB}/nccl/lib',
        f'{NVIDIA_LIB}/curand/lib',
    ]
    os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

## 1. Load Test Set

In [ ]:
from src.utils.data_loader import load_test_split
from src.evaluation.embeddings import load_test_images

paths, labels, num_classes = load_test_split('../data/test_split.json')
print(f'Test samples : {len(paths)}')
print(f'Num classes  : {num_classes}')

# Load all images into memory
images = load_test_images(paths)
print(f'Images shape : {images.shape}')
print(f'Images dtype : {images.dtype}')

## 2. Generate Genuine & Impostor Pairs

In [ ]:
from src.evaluation.pairs import generate_pairs

genuine_pairs, impostor_pairs = generate_pairs(labels, seed=42, impostor_ratio=100)
print(f'Genuine pairs  : {len(genuine_pairs)}')
print(f'Impostor pairs : {len(impostor_pairs)}')
print(f'Ratio          : 1:{len(impostor_pairs) // max(len(genuine_pairs), 1)}')

## 3. Extract Embeddings & Codes

In [ ]:
from src.evaluation.embeddings import extract_arcface_embeddings, extract_softmax_embeddings, extract_gabor_codes

# Update paths relative to notebook location
import src.evaluation.embeddings as emb_mod
emb_mod.ARCFACE_MODEL_PATH = '../models/arcface_best.h5'
emb_mod.SOFTMAX_MODEL_PATH = '../models/softmax_best.h5'

print('=== ArcFace Embeddings ===')
emb_arcface = extract_arcface_embeddings(images)
print(f'Shape: {emb_arcface.shape}\n')

print('=== Softmax Embeddings ===')
emb_softmax = extract_softmax_embeddings(images, num_classes=4115)
print(f'Shape: {emb_softmax.shape}\n')

print('=== Gabor IrisCodes ===')
codes_gabor = extract_gabor_codes(images)
print(f'Shape: {codes_gabor.shape}')

## 4. Compute Similarity Scores

In [ ]:
from src.evaluation.pairs import compute_cosine_scores, compute_hamming_scores

# ArcFace scores
gen_scores_arcface = compute_cosine_scores(emb_arcface, genuine_pairs)
imp_scores_arcface = compute_cosine_scores(emb_arcface, impostor_pairs)
print(f'ArcFace  — genuine: [{gen_scores_arcface.min():.4f}, {gen_scores_arcface.max():.4f}], '
      f'impostor: [{imp_scores_arcface.min():.4f}, {imp_scores_arcface.max():.4f}]')

# Softmax scores
gen_scores_softmax = compute_cosine_scores(emb_softmax, genuine_pairs)
imp_scores_softmax = compute_cosine_scores(emb_softmax, impostor_pairs)
print(f'Softmax  — genuine: [{gen_scores_softmax.min():.4f}, {gen_scores_softmax.max():.4f}], '
      f'impostor: [{imp_scores_softmax.min():.4f}, {imp_scores_softmax.max():.4f}]')

# Gabor scores (1 - Hamming Distance)
gen_scores_gabor = compute_hamming_scores(codes_gabor, genuine_pairs)
imp_scores_gabor = compute_hamming_scores(codes_gabor, impostor_pairs)
print(f'Gabor    — genuine: [{gen_scores_gabor.min():.4f}, {gen_scores_gabor.max():.4f}], '
      f'impostor: [{imp_scores_gabor.min():.4f}, {imp_scores_gabor.max():.4f}]')

# Bundle for plotting functions
results = {
    'ArcFace': (gen_scores_arcface, imp_scores_arcface),
    'Softmax': (gen_scores_softmax, imp_scores_softmax),
    'Gabor':   (gen_scores_gabor,   imp_scores_gabor),
}

### Diagnostic: Embedding Quality Check

Verify that embeddings from both deep learning systems show good discrimination between genuine and impostor pairs:

In [ ]:
# Embedding quality diagnostics
for name, emb, gen, imp in [
    ('ArcFace', emb_arcface, gen_scores_arcface, imp_scores_arcface),
    ('Softmax', emb_softmax, gen_scores_softmax, imp_scores_softmax),
]:
    print(f'{name} embedding statistics:')
    print(f'  Mean cosine similarity (genuine):  {gen.mean():.6f}')
    print(f'  Mean cosine similarity (impostor): {imp.mean():.6f}')
    print(f'  Genuine - Impostor gap:            {gen.mean() - imp.mean():.6f}')
    print(f'  Std across all embeddings per dim:  {emb.std(axis=0).mean():.8f}')
    print()

## 5. Comparison Table — EER & TAR

In [ ]:
from src.evaluation.plotting import build_comparison_table

comparison = build_comparison_table(results)
print(comparison.to_string())
comparison

## 6. ROC Curves

In [ ]:
from src.evaluation.plotting import plot_roc_curves
fig = plot_roc_curves(results)
plt.show()

## 7. DET Curves

In [ ]:
from src.evaluation.plotting import plot_det_curves
fig = plot_det_curves(results)
plt.show()

## 8. Score Distributions

In [ ]:
from src.evaluation.plotting import plot_score_distributions
fig = plot_score_distributions(results)
plt.show()

## 9. Training Curves

In [ ]:
from src.evaluation.plotting import plot_training_curves

history_paths = {
    'Softmax': '../models/softmax_history.json',
    'ArcFace': '../models/arcface_history.json',
}
fig = plot_training_curves(history_paths)
plt.show()

## 10. t-SNE Visualization (ArcFace Embeddings)

In [ ]:
from src.evaluation.plotting import plot_embedding_tsne

labels_arr = np.array(labels)
fig = plot_embedding_tsne(emb_arcface, labels_arr,
                          title='t-SNE — ArcFace Embeddings (top 20 identities)',
                          top_k=20)
plt.show()

## 11. Summary

### Results

| Metric | ArcFace | Softmax | Gabor |
|---|---|---|---|
| **EER** | **3.46%** | 4.20% | 26.67% |
| **TAR @ FAR=1%** | **93.33%** | 93.09% | 42.72% |
| **TAR @ FAR=0.1%** | 52.10% | **82.72%** | 29.14% |

### Key Findings

**ArcFace (EER 3.46%)** achieves the best overall verification performance. The angular margin loss produces tightly clustered, well-separated embeddings that excel at the standard operating point (FAR=1%). The training required careful configuration: SGD with momentum (not Adam), margin/scale annealing (warmup at m=0, s=16, then linear ramp to m=0.5, s=64), step LR decay at epochs 25/35/45, and filtering out single-sample identities. With these fixes, ArcFace surpasses the softmax baseline, confirming the theoretical advantage of metric learning over classification-based embeddings for verification tasks.

**Softmax (EER 4.20%)** achieves strong performance despite being trained only for closed-set classification. The L2-normalised penultimate layer produces highly discriminative embeddings. Notably, it achieves the best TAR@FAR=0.1% (82.72% vs ArcFace's 52.10%), suggesting the softmax-trained embeddings have a tighter genuine score distribution at the strictest thresholds. This is likely because softmax training with 4,115 classes provides more stable gradients than ArcFace with angular margin on this dataset (~7.5 samples per class).

**Gabor Baseline (EER 26.67%)** performs as expected for a classical hand-crafted system. It requires no training data and provides a useful reference point. The score distributions show partial separation between genuine and impostor pairs, consistent with the classical Daugman iris recognition literature.

### ArcFace Training Configuration (Critical for Reproducibility)

The initial ArcFace training collapsed (EER ~47.6%) due to five compounding issues: Adam optimizer, full margin from epoch 0, high scale from epoch 0, single-sample classes, and weak augmentation. The fix required:

1. **SGD with momentum** (0.9) + weight decay (5e-4) instead of Adam
2. **Margin/scale annealing**: 5-epoch warmup (m=0, s=16), then 15-epoch linear ramp to m=0.5, s=64
3. **Step LR decay**: 0.01 → 0.001 → 0.0001 → 0.00001 at epochs 25/35/45
4. **Class filtering**: Exclude identities with < 2 samples (3,960 classes retained)
5. **Stronger augmentation**: RandomZoom, RandomTranslation, GaussianNoise in addition to RandomRotation
6. **No early stopping**: EarlyStopping is incompatible with margin annealing (val_loss/val_accuracy are not comparable across different margin/scale settings)